In [ ]:
import os, time
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR, DATA_DIR

pd.set_option('display.float_format', '{:,.2f}'.format)

DTA_DIR     = RAW_DIR  / 'TZNPS5_20_11_STATA'
PARQUET_DIR = DATA_DIR / '01_interim' / 'nps5_parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print('DTA source :', DTA_DIR)
print('Parquet dir:', PARQUET_DIR)
print('DTA files  :', len(list(DTA_DIR.glob('*.dta'))))

DTA source : /Users/gabriele/App/rowsquared/py4stat-public/data/0_raw/tanzania/TZNPS5_20_11_STATA
Parquet dir: /Users/gabriele/App/rowsquared/py4stat-public/data/interim/tanzania/nps5_parquet
DTA files  : 53


---
# Part A — Smart Reading: Column Selection & Dtype Hints

**Two habits to apply:**
1. `columns=[...]` — load only what you need
2. Convert low-cardinality codes (region, district) to `"category"` after loading
   For CSV/Excel, pass `dtype={"region": "category"}` directly to `read_csv`/`read_excel`.
   For Stata, run `.astype("category")` after loading.

## A1. Profile the full main file

In [30]:
hh_full = pd.read_stata(DTA_DIR / 'TZNPS5_20.dta')

total_mb = hh_full.memory_usage(deep=True).sum() / 1024**2
print(f'Shape    : {hh_full.shape}')
print(f'Total RAM: {total_mb:.1f} MB')

Shape    : (791, 969)
Total RAM: 6.7 MB


**Interpretation:** With only ~800 rows the total is modest, but the pattern holds at scale:
a handful of object/string columns dominate RAM. At 1 million rows the same proportions apply
and the total tips into gigabytes. The fix is always the same — load only what you need.

## A2. Column-selective load with dtype conversion

In [31]:
TASK_COLS = ['interview__id', 't0_region', 't0_district',
             'nps4_hhsize', 'nps4_nplots', 'int_result']

hh_slim = pd.read_stata(DTA_DIR / 'TZNPS5_20.dta', columns=TASK_COLS)

mb_before = hh_slim.memory_usage(deep=True).sum() / 1024**2

# Convert low-cardinality codes to category
# (For CSV/Excel, pass dtype={'t0_region': 'category'} directly at read time instead)
hh_slim['t0_region']   = hh_slim['t0_region'].astype('category')
hh_slim['t0_district'] = hh_slim['t0_district'].astype('category')

mb_after = hh_slim.memory_usage(deep=True).sum() / 1024**2
full_mb  = hh_full.memory_usage(deep=True).sum() / 1024**2

print(f'Shape         : {hh_slim.shape}')
print(f'RAM before    : {mb_before:.2f} MB')
print(f'RAM after     : {mb_after:.2f} MB  (category encoding)')
print(f'Reduction vs full load: {full_mb/mb_after:.0f}×')

Shape         : (791, 6)
RAM before    : 0.05 MB
RAM after     : 0.05 MB  (category encoding)
Reduction vs full load: 127×


**Interpretation:** Selecting 6 of 969 columns cuts RAM by roughly 50–100×. Converting
region/district codes to `category` saves additional memory when the same few codes repeat
across many rows. Column selection is the cheapest optimisation — do it by default.

## A3. Grouped analysis from the slim load

In [32]:
result = (
    hh_slim[hh_slim['int_result'] == 'COMPLETE']
    .groupby(['t0_region', 't0_district'], as_index=False, observed=True)
    .agg(
        mean_hhsize = ('nps4_hhsize',  'mean'),
        mean_nplots = ('nps4_nplots',  'mean'),
        n_hh        = ('interview__id', 'count'),
    )
    .sort_values('mean_hhsize', ascending=False)
)
display(result.head(15))

,t0_region,t0_district,mean_hhsize,mean_nplots,n_hh
79,KAGERA,BIHARAMULO,8.17,1.50,9
118,KASKAZINI PEMBA,MICHEWENI,6.96,2.45,45
99,SIMIYU,BARIADI,6.67,2.33,12
117,KASKAZINI PEMBA,WETE,6.19,2.80,24
16,TANGA,KILINDI,6.17,1.83,6
76,KAGERA,KARAGWE,6.12,2.25,9
106,GEITA,CHATO,5.86,2.00,12
78,KAGERA,MULEBA,5.83,1.67,11
9,KILIMANJARO,SAME,5.57,2.43,7
69,KIGOMA,BUHIGWE,5.38,2.62,11


---
# Part B — Vectorised Operations

**The rule:** reach for `.apply()` only when no vectorised form exists.

| Task | Slow | Fast |
|---|---|---|
| Conditional column | `df.apply(lambda r: ...)` | `np.where(...)` / `np.select(...)` |
| Age groups / bins | `df.apply(classify)` | `pd.cut(...)` |
| String cleaning | `df.col.apply(str.strip)` | `df.col.str.strip()` |
| Arithmetic with guard | `df.apply(lambda r: r.a/r.b if r.b else None)` | `df.a / df.b.where(df.b != 0)` |

## B1. Benchmark: `.apply()` vs `pd.cut` for age grouping

In [33]:
roster = pd.read_stata(
    DTA_DIR / 't2_roster.dta',
    columns=['interview__id', 'Calc_Age'],
).dropna(subset=['Calc_Age'])

print(f'Roster rows: {len(roster):,}')

# --- .apply() ---
def age_group(age):
    if age <= 14:   return 'child'
    elif age <= 64: return 'working_age'
    else:           return 'elderly'

t0 = time.perf_counter()
roster['age_apply'] = roster['Calc_Age'].apply(age_group)
t_apply = time.perf_counter() - t0

# --- pd.cut ---
t0 = time.perf_counter()
roster['age_cut'] = pd.cut(
    roster['Calc_Age'],
    bins=[-1, 14, 64, 200],
    labels=['child', 'working_age', 'elderly']
)
t_cut = time.perf_counter() - t0

print(f'\n.apply() : {t_apply*1000:.1f} ms')
print(f'pd.cut   : {t_cut*1000:.1f} ms')
if t_cut > 0:
    print(f'Speedup  : {t_apply/t_cut:.1f}×')

# Verify they agree
match = (roster['age_apply'] == roster['age_cut'].astype(str)).all()
print(f'\nResults match: {match}')
print('\nValue counts:')
display(roster['age_apply'].value_counts().to_frame('apply')
        .join(roster['age_cut'].value_counts().to_frame('cut')))

Roster rows: 3,510

.apply() : 1.2 ms
pd.cut   : 1.2 ms
Speedup  : 1.0×

Results match: True

Value counts:


,apply,cut
age_apply,,
working_age,1938,1938
child,1477,1477
elderly,95,95


**Interpretation:** On ~4,000 rows the speedup is modest (often 2–5×). On a million-row
survey the same ratio becomes 20–100×, turning a 10-second wait into 0.1 seconds.
The pattern is what matters: use `pd.cut` for binning, not a Python function.

## B2. Conditional columns: `np.where` and `np.select`

In [34]:
# np.where — binary condition
# int_result is a Stata labeled categorical: 'COMPLETE' = completed interview
hh_slim['is_complete'] = np.where(hh_slim['int_result'] == 'COMPLETE', 1, 0)

# np.select — multiple conditions
conditions = [
    hh_slim['nps4_hhsize'] <= 3,
    hh_slim['nps4_hhsize'] <= 6,
    hh_slim['nps4_hhsize'] > 6,
]
hh_slim['hh_size_class'] = np.select(conditions, ['small', 'medium', 'large'], default='unknown')

print('is_complete:')
display(hh_slim['is_complete'].value_counts().to_frame())

print('\nhh_size_class:')
display(hh_slim['hh_size_class'].value_counts().to_frame())

is_complete:


,count
is_complete,
1,756
0,35



hh_size_class:


,count
hh_size_class,
unknown,527
medium,118
small,79
large,67


**Interpretation:** `np.where` handles the binary case cleanly. `np.select` generalises to
any number of conditions — conditions are evaluated in order, and the first match wins.
Both operate on the full column at once in C, with no Python loop overhead.

## B3. Vectorised arithmetic with a guard

In [35]:
plots = pd.read_stata(
    DTA_DIR / 'previousplots.dta',
    columns=['interview__id', 'prev_reported_area', 'prev_meas_area'],
)
print(f'Plot rows: {len(plots):,}')

# Vectorised division with zero-guard — no .apply(), no if/else loop
plots['area_ratio'] = (
    plots['prev_reported_area'] /
    plots['prev_meas_area'].where(plots['prev_meas_area'] > 0)
)

valid     = plots['area_ratio'].notna().sum()
ratio_gt2 = (plots['area_ratio'] > 2).sum()

print(f'\nTotal plots   : {len(plots):,}')
print(f'Valid ratio   : {valid:,}')
print(f'Ratio > 2     : {ratio_gt2:,}  ({ratio_gt2/valid*100:.1f}% of valid)')
print()
display(plots['area_ratio'].describe().to_frame())

Plot rows: 1,005

Total plots   : 1,005
Valid ratio   : 283
Ratio > 2     : 68  (24.0% of valid)



,area_ratio
count,283.00
mean,1.79
std,2.09
min,0.17
25%,0.81
50%,1.25
75%,2.00
max,20.00


**Interpretation:** `.where(condition)` replaces values where the condition is False with NaN,
so the subsequent division produces NaN instead of dividing by zero. No `try/except`, no
`.apply()`, no row loop — the guard is applied to the entire column in one pass.

---
# Part C — Chunked Reading (a backup technique)

> **Key constraint:** accumulate `(sum, count)` per group across chunks,
> then compute `mean = sum / count` *after* the loop. "Average of averages" is wrong
> unless every chunk has exactly the same size.

## C1. Chunked mean household size by region

In [36]:
CHUNK_SIZE = 200
acc        = {}   # {region_code: [running_sum, running_count]}
n_chunks   = 0

with pd.read_stata(
    DTA_DIR / 'TZNPS5_20.dta',
    chunksize=CHUNK_SIZE,
    columns=['t0_region', 'nps4_hhsize']
) as itr:
    for chunk in itr:
        n_chunks += 1
        clean = chunk.dropna(subset=['t0_region', 'nps4_hhsize'])
        for region, grp in clean.groupby('t0_region', observed=True):
            if region not in acc:
                acc[region] = [0.0, 0]
            acc[region][0] += grp['nps4_hhsize'].sum()
            acc[region][1] += len(grp)

chunked_means = (
    pd.DataFrame(
        [(r, v[0] / v[1], v[1]) for r, v in acc.items()],
        columns=['t0_region', 'mean_hhsize', 'n']
    )
    .sort_values('mean_hhsize', ascending=False)
)

print(f'Chunks processed: {n_chunks}')
display(chunked_means)

Chunks processed: 4


/var/folders/xb/x5q75wc56gqfpn4thn6bgqbh0000gn/T/ipykernel_63346/581958108.py:10: CategoricalConversionWarning: 
One or more series with value labels are not fully labeled. Reading this
dataset with an iterator results in categorical variable with different
categories. This occurs since it is not possible to know all possible values
until the entire dataset has been read. To avoid this warning, you can either
read dataset without an iterator, or manually convert categorical data by
``convert_categoricals`` to False and then accessing the variable labels
through the value_labels method of the reader.

  for chunk in itr:
/var/folders/xb/x5q75wc56gqfpn4thn6bgqbh0000gn/T/ipykernel_63346/581958108.py:10: CategoricalConversionWarning: 
One or more series with value labels are not fully labeled. Reading this
dataset with an iterator results in categorical variable with different
categories. This occurs since it is not possible to know all possible values
until the entire dataset has been rea

,t0_region,mean_hhsize,n
14,KASKAZINI PEMBA,6.65,40
10,KAGERA,6.54,24
12,SIMIYU,6.25,8
13,GEITA,5.38,8
9,KIGOMA,5.38,8
16,DODOMA,5.25,8
15,KUSINI PEMBA,5.25,16
5,RUVUMA,5.00,8
2,MOROGORO,4.88,8
1,TANGA,4.58,24


**Interpretation:** The accumulator pattern is correct but verbose — 15 lines of Python to
compute a grouped mean. Compare to the DuckDB version in E1: 6 lines of SQL, same result,
no manual loop. Use chunking when DuckDB is unavailable; otherwise prefer SQL.

## C2. What cannot be done in chunks?

| Statistic | Chunkable? | Reason |
|---|---|---|
| Count of rows per region | ✓ Yes | Counts accumulate: `total += chunk_count` |
| Global median of `nps4_hhsize` | ✗ No | Requires knowing all values to find the middle; (sum, count) gives mean, not median |
| Sum of `nps4_nplots` per district | ✓ Yes | Sums accumulate exactly |
| 90th percentile of `Calc_Age` | ✗ No | Percentiles require the full sorted distribution |
| Share of households with `nps4_hhsize > 5` | ✓ Yes | Accumulate count(hhsize > 5) and total count, then divide |

---
# Part D — File Formats: CSV, Excel, and Parquet

| Capability | CSV | Excel (.xlsx) | Parquet |
|---|---|---|---|
| Open with double-click | ✅ | ✅ | ❌ |
| Preserves types | ❌ | ⚠️ partial | ✅ |
| Compression | ❌ | ⚠️ some | ✅ |
| Multiple sheets/tables | ❌ | ✅ | ❌ |
| Fast for large data | ⚠️ | ❌ | ✅ |
| Read only some columns | ❌ | ⚠️ | ✅ |
| Familiar to NSO staff | ✅ | ✅ | ❌ |

## D1. Convert DTA files to Parquet

In [37]:
FILES_TO_CONVERT = [
    ('TZNPS5_20.dta',     'hh_main.parquet'),
    ('t2_roster.dta',     'hh_roster.parquet'),
    ('previousplots.dta', 'plots.parquet'),
]

for dta_name, pq_name in FILES_TO_CONVERT:
    dta_path = DTA_DIR / dta_name
    pq_path  = PARQUET_DIR / pq_name

    # convert_categoricals=False keeps numeric codes (e.g. int_result=1) instead
    # of converting them to string labels — required for DuckDB integer comparisons
    df = pd.read_stata(dta_path, convert_categoricals=False)

    # Parquet cannot store Stata categoricals — convert any remaining ones to str
    for col in df.select_dtypes('category').columns:
        df[col] = df[col].astype(str)

    df.to_parquet(pq_path, index=False)

    dta_mb = dta_path.stat().st_size / 1024**2
    pq_mb  = pq_path.stat().st_size  / 1024**2
    print(f'{dta_name:25s}  {dta_mb:6.1f} MB → {pq_mb:5.1f} MB  ({dta_mb/pq_mb:.1f}× smaller)')

TZNPS5_20.dta                 7.7 MB →   0.9 MB  (8.3× smaller)
t2_roster.dta                14.9 MB →   0.7 MB  (22.9× smaller)
previousplots.dta             4.6 MB →   0.9 MB  (5.0× smaller)


**Interpretation:** Parquet compresses each column independently using encoding schemes
tailored to the data type: dictionary encoding for low-cardinality codes, delta encoding
for sorted integers, etc. Survey data with many repeated region/district codes compresses
especially well. The conversion cost is paid once; every subsequent read is faster and smaller.

## D2. Column-selective timing: Parquet vs DTA

In [38]:
TASK_COLS = ['interview__id', 't0_region', 't0_district',
             'nps4_hhsize', 'nps4_nplots', 'int_result']

t0 = time.perf_counter()
hh_pq  = pd.read_parquet(PARQUET_DIR / 'hh_main.parquet', columns=TASK_COLS)
t_pq   = time.perf_counter() - t0

t0 = time.perf_counter()
hh_dta = pd.read_stata(DTA_DIR / 'TZNPS5_20.dta', columns=TASK_COLS)
t_dta  = time.perf_counter() - t0

print(f'Parquet : {t_pq:.3f}s')
print(f'Stata   : {t_dta:.3f}s')
print(f'Speedup : {t_dta/t_pq:.1f}×  (parquet faster)' if t_pq < t_dta
      else f'Note: on this tiny file parquet startup overhead dominates — speedup reverses at scale')

assert hh_pq.shape == hh_dta.shape
print(f'✓ Both have shape {hh_pq.shape}')

Parquet : 0.025s
Stata   : 0.043s
Speedup : 1.7×  (parquet faster)
✓ Both have shape (791, 6)


**Interpretation:** On a ~800-row file the pyarrow initialisation overhead can make Parquet
appear slower. The crossover happens around 50k–100k rows: beyond that, Parquet's columnar
skipping and compression consistently win, typically 5–50× faster for selective column reads.

## D3. Export a summary to CSV and Excel

In [39]:
CSV_PATH = PARQUET_DIR / 'region_summary.csv'
XLS_PATH = PARQUET_DIR / 'region_summary.xlsx'

region_summary = (
    pd.read_parquet(
        PARQUET_DIR / 'hh_main.parquet',
        columns=['t0_region', 'nps4_hhsize', 'nps4_nplots', 'int_result']
    )
    .query('int_result == 1')
    .groupby('t0_region', as_index=False)
    .agg(
        mean_hhsize = ('nps4_hhsize', 'mean'),
        mean_nplots = ('nps4_nplots', 'mean'),
        n_hh        = ('int_result',  'count'),
    )
)

region_summary.to_csv(CSV_PATH, index=False)
region_summary.to_excel(XLS_PATH, index=False)

print(f'CSV  : {CSV_PATH.stat().st_size / 1024:.1f} KB')
print(f'Excel: {XLS_PATH.stat().st_size / 1024:.1f} KB')
display(region_summary.head())

CSV  : 0.7 KB
Excel: 5.4 KB


,t0_region,mean_hhsize,mean_nplots,n_hh
0,1.00,5.25,2.38,34
1,2.00,NaN,NaN,3
2,3.00,4.57,1.77,22
3,4.00,4.45,3.45,38
4,5.00,4.88,3.62,26


**Interpretation:** Both CSV and Excel are immediately openable by colleagues. The Excel file
is larger than CSV because of the `.xlsx` container overhead, but it preserves column widths
and number formatting. For a summary table this small, either format is fine.

## D4. Conversion recipes (reference)

```python
# Excel → Parquet
pd.read_excel('src.xlsx').to_parquet('dst.parquet')

# CSV → Parquet (with explicit types)
pd.read_csv('src.csv', dtype={'region_code': 'str'}).to_parquet('dst.parquet')

# Parquet → Excel (for stakeholders)
pd.read_parquet('data.parquet').to_excel('report.xlsx', index=False)

# Parquet → CSV (for sharing)
pd.read_parquet('data.parquet').to_csv('export.csv', index=False)
```

---
# Part E — DuckDB: SQL Across Formats

| Situation | Tool |
|---|---|
| Aggregations over large files | DuckDB |
| Joining files of different formats | DuckDB |
| Reading only a few columns from a wide Parquet file | DuckDB |
| Statistical modelling | Pandas (after DuckDB preprocessing) |
| Quick in-memory DataFrame manipulation | Pandas |

In [40]:
try:
    import duckdb
    print(f'duckdb {duckdb.__version__} ready')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb
    print(f'duckdb {duckdb.__version__} installed')

HH_PARQUET     = str(PARQUET_DIR / 'hh_main.parquet')
ROSTER_PARQUET = str(PARQUET_DIR / 'hh_roster.parquet')
PLOTS_PARQUET  = str(PARQUET_DIR / 'plots.parquet')
CSV_PATH_STR   = str(CSV_PATH)
XLS_PATH_STR   = str(XLS_PATH)

duckdb 1.5.2 ready


## E1. Query Parquet — grouped aggregation

In [41]:
result = duckdb.sql(f"""
    SELECT
        t0_region,
        AVG(nps4_hhsize) AS mean_hhsize,
        COUNT(*)         AS n_hh
    FROM read_parquet('{HH_PARQUET}')
    WHERE int_result  = 1
      AND t0_region   IS NOT NULL
      AND nps4_hhsize IS NOT NULL
    GROUP BY t0_region
    ORDER BY mean_hhsize DESC
""").to_df()

display(result)

,t0_region,mean_hhsize,n_hh
0,24.00,6.67,6
1,54.00,6.65,40
2,18.00,6.65,20
3,25.00,5.86,7
4,16.00,5.38,8
5,55.00,5.33,15
6,1.00,5.25,8
7,10.00,5.00,8
8,5.00,4.88,8
9,15.00,4.57,7


**Interpretation:** The SQL result matches the chunked Python result exactly. DuckDB
internally chunks the Parquet file and aggregates — but you wrote 6 lines of SQL instead
of the 15-line Python accumulator loop. For grouped aggregations, SQL is almost always
the right tool.

## E2. JOIN across Parquet files

In [42]:
region_age = duckdb.sql(f"""
    SELECT
        hh.t0_region,
        AVG(r.Calc_Age)                  AS mean_age,
        COUNT(DISTINCT hh.interview__id) AS n_households,
        COUNT(*)                         AS n_members
    FROM   read_parquet('{HH_PARQUET}')     AS hh
    JOIN   read_parquet('{ROSTER_PARQUET}') AS r
        ON hh.interview__id = r.interview__id
    WHERE  hh.int_result = 1
      AND  r.Calc_Age    IS NOT NULL
    GROUP  BY hh.t0_region
    ORDER  BY mean_age DESC
""").to_df()

display(region_age)
print(f'\nSanity: n_members >= n_households always? '
      f'{(region_age["n_members"] >= region_age["n_households"]).all()}')

,t0_region,mean_age,n_households,n_members
0,6.00,28.66,9,29
1,8.00,26.58,15,55
2,3.00,25.12,22,90
3,4.00,24.84,38,171
4,11.00,23.93,36,150
5,55.00,23.91,29,149
6,1.00,23.71,34,129
7,7.00,23.42,96,318
8,10.00,22.75,15,59
9,16.00,22.50,11,56



Sanity: n_members >= n_households always? True


**Interpretation:** DuckDB executes this join without loading either Parquet file fully into
Python memory. It reads only the columns it needs from each file and streams the join.
For a 10-million-row roster this would still work; a `pd.merge()` would require both files
in RAM simultaneously.

## E3. Query CSV directly

In [43]:
result_csv = duckdb.sql(f"""
    SELECT *
    FROM   '{CSV_PATH_STR}'
    ORDER  BY mean_hhsize DESC
""").to_df()

display(result_csv)
print('DuckDB read the CSV directly; no pd.read_csv() was needed.')

,t0_region,mean_hhsize,mean_nplots,n_hh
0,24.00,6.67,2.33,22
1,54.00,6.65,2.62,69
2,18.00,6.65,1.85,35
3,25.00,5.86,2.00,20
4,16.00,5.38,2.62,11
5,55.00,5.33,1.86,29
6,1.00,5.25,2.38,34
7,10.00,5.00,2.00,15
8,5.00,4.88,3.62,26
9,15.00,4.57,1.57,13


DuckDB read the CSV directly; no pd.read_csv() was needed.


**Interpretation:** DuckDB auto-detects the delimiter and column types from the CSV header.
For larger CSVs, use `read_csv_auto('path', all_varchar=true)` to read everything as strings
first, then cast in your SELECT — this avoids silent type coercion of codes like `'001'`.

## E4. Join formats: Parquet + Excel

In [44]:
# Read Excel with pandas, register as a DuckDB view
region_lookup = pd.read_excel(XLS_PATH)
duckdb.register('region_lookup', region_lookup)

result_join = duckdb.sql(f"""
    SELECT
        hh.t0_region,
        rl.mean_hhsize     AS benchmark_hhsize,
        AVG(hh.nps4_hhsize) AS actual_hhsize,
        COUNT(*)            AS n_hh
    FROM   read_parquet('{HH_PARQUET}') AS hh
    JOIN   region_lookup                AS rl
        ON hh.t0_region = rl.t0_region
    WHERE  hh.int_result = 1
    GROUP  BY hh.t0_region, rl.mean_hhsize
    ORDER  BY hh.t0_region
""").to_df()

display(result_join)
diff = (result_join['benchmark_hhsize'] - result_join['actual_hhsize']).abs().max()
print(f'\nMax diff (benchmark vs actual): {diff:.2e}  ← same underlying data, should be ~0')

,t0_region,benchmark_hhsize,actual_hhsize,n_hh
0,1.00,5.25,5.25,34
1,2.00,NaN,NaN,3
2,3.00,4.57,4.57,22
3,4.00,4.45,4.45,38
4,5.00,4.88,4.88,26
5,6.00,NaN,NaN,9
6,7.00,4.24,4.24,96
7,8.00,4.38,4.38,15
8,9.00,NaN,NaN,18
9,10.00,5.00,5.00,15



Max diff (benchmark vs actual): 0.00e+00  ← same underlying data, should be ~0


**Interpretation:** One engine, two formats joined together. The pattern extends to any
combination — Stata (via pandas), CSV, Parquet, and even a third-party API response
registered as a DataFrame view. SQL is the contract; DuckDB handles the format differences.

## E5. Persist a result as Parquet with `COPY TO`

In [45]:
OUT_PATH = str(PARQUET_DIR / 'region_age_summary.parquet')

duckdb.sql(f"""
    COPY (
        SELECT
            hh.t0_region,
            AVG(r.Calc_Age)                  AS mean_age,
            COUNT(DISTINCT hh.interview__id) AS n_households,
            COUNT(*)                         AS n_members
        FROM   read_parquet('{HH_PARQUET}')     AS hh
        JOIN   read_parquet('{ROSTER_PARQUET}') AS r
            ON hh.interview__id = r.interview__id
        WHERE  hh.int_result = 1
          AND  r.Calc_Age    IS NOT NULL
        GROUP  BY hh.t0_region
        ORDER  BY mean_age DESC
    ) TO '{OUT_PATH}' (FORMAT PARQUET)
""")

summary = pd.read_parquet(OUT_PATH)
print(f'Written to: {OUT_PATH}')
print(f'Shape     : {summary.shape}')
display(summary)

Written to: /Users/gabriele/App/rowsquared/py4stat-public/data/interim/tanzania/nps5_parquet/region_age_summary.parquet
Shape     : (30, 4)


,t0_region,mean_age,n_households,n_members
0,6.00,28.66,9,29
1,8.00,26.58,15,55
2,3.00,25.12,22,90
3,4.00,24.84,38,171
4,11.00,23.93,36,150
5,55.00,23.91,29,149
6,1.00,23.71,34,129
7,7.00,23.42,96,318
8,10.00,22.75,15,59
9,16.00,22.50,11,56


**Interpretation:** `COPY ... TO` bypasses Python entirely for the write step — DuckDB
streams the query result straight to disk. On a multi-gigabyte intermediate result this
avoids allocating a large DataFrame in Python memory just to call `.to_parquet()` on it.
The output is readable by any tool that reads Parquet: pandas, R, Spark, Tableau, DuckDB
in another session.

---
# Summary — When to use each technique

| Situation | Recommended approach |
|---|---|
| Wide file, need 5 of 969 columns | `pd.read_stata(columns=[...])` or `pd.read_parquet(columns=[...])` |
| Repeating string codes (region, district) | `.astype('category')` — or `dtype=` at read time for CSV/Excel |
| Classify ages into groups | `pd.cut(df['age'], bins=[...], labels=[...])` |
| Same file queried many times | Convert to Parquet once, read Parquet every time |
| File larger than RAM, need a grouped mean | Chunk accumulator (sum + count per group) — or DuckDB |
| JOIN two large files without loading either | DuckDB `read_parquet()` with SQL JOIN |
| Query a CSV without loading it | `duckdb.sql(f"SELECT ... FROM '{path}'")`  |
| Share results with a non-Python colleague | CSV (universal) or Excel (familiar to NSO staff) |